# 标记笔记本
你可以通过运行这个笔记本，按照流程标记你的目标

In [ ]:
from pathlib import Path
import re

import numpy as np
from numpy.typing import NDArray
from scipy import ndimage as nd 
from tifffile import imread, imwrite
import napari
from loguru import logger
from tqdm import tqdm
import typer

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from loguru import logger

# Load environment variables from .env file if it exists
load_dotenv()

# Paths
PROJ_ROOT = Path("notebooks").resolve().parents[1]
logger.info(f"PROJ_ROOT path is: {PROJ_ROOT}")

DATA_DIR = PROJ_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
IMAGES_DATA_DIR = DATA_DIR / "images"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
EXTERNAL_DATA_DIR = DATA_DIR / "external"

TRAIN_DATA_DIR = PROCESSED_DATA_DIR / "train"
TEST_DATA_DIR = PROCESSED_DATA_DIR / "test"
MODELS_DIR = PROJ_ROOT / "models"

REPORTS_DIR = PROJ_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

# If tqdm is installed, configure loguru with tqdm.write
# https://github.com/Delgan/loguru/issues/135
try:
    from tqdm import tqdm

    logger.remove(0)
    logger.add(lambda msg: tqdm.write(msg, end=""), colorize=True)
except ModuleNotFoundError:
    pass

In [ ]:
# Set Path
input_path: Path = IMAGES_DATA_DIR
output_path: Path = EXTERNAL_DATA_DIR

In [ ]:
logger.info("Processing dataset...")
logger.info(f"Input path: {input_path}")
logger.info(f"Output path: {output_path}")
# Load image-------------------------------------
tmp_list: list[Path] = list(Path(input_path).iterdir())
img_list: list[Path] = []
for tmp in tmp_list:
    if re.search('.tif', str(tmp)):
        img_list.append(tmp) # img_list have full path, not name, e.g. "/path/to/img.tif"
        
logger.info("images found:\n"+"\n".join(map(lambda x: str(x.name), img_list)))
logger.info(f"Number of images found: {len(img_list)}")

## 这里需要你指定`img_index`
只需要指定一次，最后一个单元会自动加`1`

提示:
* 这里的`img_index`指的是`img_list`的元素对象下标
* python列表类型的元素下标从`0`开始，到`len(list-object)-1`

In [ ]:
# Select image and load it
img_index = 0

## 现在，你可以循环执行以下单元格，逐图进行标记

In [ ]:

logger.success(f"Selected image: {img_list[img_index].name}")
img_path: Path = img_list[img_index] # img_list have full path
logger.info(f"Image path: {img_path}")
img = imread(img_path)
logger.info(img.shape)

## 指定通道序号
务必确定每个通道的含义并填写指定通道序号，辅助工具：FIJI

In [ ]:
viewer = napari.Viewer()

# 务必确定每个通道的含义，并填写指定通道序号，辅助工具：FIJI
actin_ch_number = int("3") - 1
mito_ch_number = int("1") - 1
lipid_ch_number = int("2") - 1

actin_img = img[actin_ch_number, :, :]
mito_img = img[mito_ch_number, :, :]
lipid_img = img[lipid_ch_number, :, :]

viewer.add_image(actin_img, name="actin")
viewer.add_image(mito_img, name="mito")
viewer.add_image(lipid_img, name="lipid")

## 接下来，需要你手动操作画笔，标注以下内容：
* roi遮罩层：名称，`roi - Labels`，涂抹的区域是将是模型真正工作的区域，涂抹区域外无论是否有目标都将视为背景。
    特殊情况，若不涂抹，默认全图识别
* 线粒体标记：名称，`mito - Labels`，标记线粒体
* 脂滴标记：名称，`lipid - Labels`，标记脂滴

保证名称一致
标注完毕后，再运行剩余单元格

In [ ]:
# 你需要手动标记需要识别的区域
roi_mask = viewer.layers['roi - Labels'].data
# 你需要手动标记mito
mito_mask = viewer.layers['mito - Labels'].data
# 你需要手动标记lipid
lipid_mask = viewer.layers['lipid - Labels'].data

In [ ]:
# Mask work area, save _masks.tif and _roi.tif in /data/external
assert mito_mask.shape == lipid_mask.shape
masks = np.stack([mito_mask, lipid_mask], axis=0).astype(np.uint16)
masks_path = output_path / f"{img_list[img_index].name.replace(".tif", "_masks.tif")}"
roi_path = output_path / f"{img_list[img_index].name.replace(".tif", "_roi.tif")}"
logger.info(f"roi_shape: {roi_mask.shape}")
logger.info(f"masks_shape: {masks.shape}")
imwrite(masks_path, np.uint16(masks))
imwrite(roi_path, np.uint16(roi_mask))

In [ ]:
viewer.close()
if img_index < len(img_list):
    logger.info(f"第{img_index + 1}张图标注完毕, 还有{len(img_list) - img_index - 1}")
    img_index += 1
else:
    logger.success("所有图片均标注完毕")